# Kaggle Utilities — Project Ouroboros

This notebook replaces the manual `/kaggle/input/...` copy-and-rename flow. It can:

- load Kaggle secrets safely
- hydrate the persistent Hugging Face cache from the attached `ouroboros-cache` dataset
- install the cached Mamba wheels
- pull the latest repo files from GitHub into `/kaggle/working`
- inspect local vs Hugging Face checkpoints
- launch Stage 1 with the latest local-or-HF checkpoint selection logic now built into `pretrain.py`
- launch Stage 2 with auto-detected Dual T4 DDP, timeout-safe checkpoint exit, and Stage-2-first resume logic built into `train_sft.py`


In [1]:
import os
import shutil
from pathlib import Path

!cp -r /kaggle/input/datasets/weirdrunner007/ouroboros-cache/* /kaggle/working/

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

In [2]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient


def optional_secret(client: UserSecretsClient, name: str) -> str | None:
    try:
        value = client.get_secret(name)
    except Exception:
        return None
    return value or None


user_secrets = UserSecretsClient()
HF_TOKEN = optional_secret(user_secrets, "HF_TOKEN")
WANDB_KEY = optional_secret(user_secrets, "WANDB_KEY")
GITHUB_TOKEN = optional_secret(user_secrets, "GITHUB_TOKEN") or optional_secret(user_secrets, "GH_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if WANDB_KEY:
    os.environ["WANDB_KEY"] = WANDB_KEY
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print({
    "HF_TOKEN": bool(HF_TOKEN),
    "WANDB_KEY": bool(WANDB_KEY),
    "GITHUB_TOKEN": bool(GITHUB_TOKEN),
})


{'HF_TOKEN': True, 'WANDB_KEY': True, 'GITHUB_TOKEN': True}


In [3]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = os.environ.get("OUROBOROS_REPO_URL", "https://github.com/deveshpat/Ouroboros.git")
REPO_REF = os.environ.get("OUROBOROS_REPO_REF", "main")
REPO_DIR = Path("/kaggle/working/ouroboros_repo")
TARGET_DIR = Path("/kaggle/working")
FILES_TO_COPY = [
    "jamba_coconut_finetune.py",
]


def run(cmd: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True, text=True, capture_output=True)


def authenticated_repo_url(repo_url: str) -> str:
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if not token:
        return repo_url
    if not repo_url.startswith("https://"):
        return repo_url
    if "github.com" not in repo_url:
        return repo_url
    if "@" in repo_url:
        return repo_url
    return f"https://x-access-token:{token}@{repo_url[len('https://') :]}"


def sync_repo(repo_url: str, ref: str, repo_dir: Path) -> str:
    auth_url = authenticated_repo_url(repo_url)
    if not repo_dir.exists():
        run(["git", "clone", "--filter=blob:none", "--branch", ref, auth_url, str(repo_dir)])
    else:
        run(["git", "fetch", "origin", ref], cwd=repo_dir)
        run(["git", "checkout", "-B", ref, "FETCH_HEAD"], cwd=repo_dir)
        run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=repo_dir)
        run(["git", "clean", "-fd"], cwd=repo_dir)
    commit = run(["git", "rev-parse", "HEAD"], cwd=repo_dir).stdout.strip()
    return commit


def copy_repo_files(repo_dir: Path, target_dir: Path, file_names: list[str]) -> list[str]:
    copied: list[str] = []
    for name in file_names:
        src = repo_dir / name
        if not src.exists():
            print(f"[warn] missing in repo: {name}")
            continue
        dst = target_dir / src.name
        shutil.copy2(src, dst)
        copied.append(dst.name)
    return copied


commit = sync_repo(REPO_URL, REPO_REF, REPO_DIR)
copied = copy_repo_files(REPO_DIR, TARGET_DIR, FILES_TO_COPY)
metadata = {
    "repo_ref": REPO_REF,
    "commit": commit,
    "copied_files": copied,
    "synced_at_utc": datetime.now(timezone.utc).isoformat(),
}
(Path("/kaggle/working/repo_sync_metadata.json")).write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))


{
  "repo_ref": "main",
  "commit": "f3c1ddcdd7d3467cf04780e5fecb837ef6b341b3",
  "copied_files": [
    "jamba_coconut_finetune.py"
  ],
  "synced_at_utc": "2026-04-21T00:51:08.284351+00:00"
}


In [4]:
# Optional: login to Weights & Biases only when the secret exists.
import os
import wandb

wandb_key = os.environ.get("WANDB_KEY")
if wandb_key:
    wandb.login(key=wandb_key)
    print("wandb login: OK")
else:
    print("wandb login: skipped (WANDB_KEY not set)")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: weirdrunner-36094 (weirdrunner-36094-ouro) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb login: OK


In [5]:
# ── Cell 5 — DiLoCo Worker Training ─────────────────────────────────────────
#
# Each account sets a Kaggle secret DILOCO_WORKER_ID = "A" / "B" / "C".
# This cell reads that secret, validates it, then launches the correct worker.
# Using subprocess.run (not !torchrun magic) so the f-string expands correctly.

import os
import subprocess
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
WORKER_ID = _secrets.get_secret("DILOCO_WORKER_ID").strip().upper()
assert WORKER_ID in ("A", "B", "C"), (
    f"DILOCO_WORKER_ID secret must be 'A', 'B', or 'C' — got {WORKER_ID!r}. "
    "Set it under Kaggle → Your Profile → Settings → Secrets."
)
print(f"[cell5] Launching as Worker {WORKER_ID}")

!torchrun --standalone --nproc_per_node=2 jamba_coconut_finetune.py \
    --data_dir data/coconut_v1 --use_4bit \
    --stage_0_epochs 1 --epochs_per_stage 1 --max_stage 10 \
    --batch_size 4 --grad_accum 8 \
    --val_batch_size 2 \
    --val_skip_buffer_minutes 60 \
    --session_timeout_hours 12.0 --graceful_exit_buffer_minutes 20 \
    --diloco_mode \
    --diloco_worker_id $WORKER_ID \
    --diloco_outer_lr 0.7 \
    --diloco_state_repo WeirdRunner/Ouroboros \
    --diloco_signal_repo deveshpat/Ouroboros \
    --push_to_hub \
    --output_dir runs/diloco \
 #   --output_dir runs/stage3_curriculum \
 #   --wandb_mode offline

[cell5] Launching as Worker B
W0421 00:51:19.183000 68 torch/distributed/run.py:852] 
W0421 00:51:19.183000 68 torch/distributed/run.py:852] *****************************************
W0421 00:51:19.183000 68 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0421 00:51:19.183000 68 torch/distributed/run.py:852] *****************************************
[W421 00:51:19.645129967 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[bootstrap][rank=1] Waiting for rank 0 shared bootstrap install...
[bootstrap] DDP guard: rank 0 performing shared bootstrap install; other ranks will wait.
[bootstrap] Phase 1: pure-Python deps...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
[bootstrap] Phase 2: arch-aware Hub wheel install...
[bootstrap]   G